# MIDI Exact-Control Music Pilot

This notebook demonstrates a **note-first, editable music workflow**. We define a short motif as explicit MIDI events, then render it into separate lead and bass layers.

## What this proves
- Every note pitch is explicit and can be constrained to a selected scale.
- Note onsets and lengths are explicit beat positions, so the rhythm is reproducible.
- Tempo, key/scale, note events, and instrument tracks can be edited independently.
- The same MIDI can be rendered with different diagnostic timbres without changing its notes.

## What this does not prove
The built-in renderer below uses simple mathematical waveforms. It is intentionally a control/architecture proof, **not a production-quality instrument**. No audio model is trained here. Later, we can replace this renderer with a commercially cleared instrument library or compare it against Stable Audio. Stable Audio remains a free-form quality/variation reference; this notebook is the exact-control baseline.

## Step 1 — Environment

This is a lightweight CPU notebook. It does not load a large checkpoint, need a GPU, download a dataset, or ask for an API key.

It uses NumPy, SciPy, and Matplotlib, which are available in the standard Colab runtime. The MIDI Standard File is written directly in Python, so there is no MIDI package installation.

In [ ]:
import json
import math
import struct
import zipfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.io.wavfile import write as write_wav
from IPython.display import Audio, display

OUT_DIR = Path("/content/midi_exact_control_pilot")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output folder:", OUT_DIR)

## Step 2 — Choose musical controls

Change these values to explore the controls:
- `BPM`: tempo in quarter notes per minute.
- `KEY_TONIC_MIDI`: tonic pitch. C4 = 60, D4 = 62.
- `SCALE_INTERVALS`: semitone offsets within one octave.
- `METER`: time signature.
- `BARS`: motif length.

The motif below is expressed in **scale degrees and beat positions**, rather than being inferred from a text prompt. This makes the control path inspectable.

In [ ]:
# Main composition settings
BPM = 96
KEY_NAME = "C"
SCALE_NAME = "major"
KEY_TONIC_MIDI = 60             # C4
SCALE_INTERVALS = [0, 2, 4, 5, 7, 9, 11]  # major scale
METER = (4, 4)
BARS = 4
TICKS_PER_BEAT = 480
SAMPLE_RATE = 44100

# Same degree pattern in a second key/scale and tempo for a controlled comparison.
ALT_BPM = 120
ALT_KEY_NAME = "D"
ALT_SCALE_NAME = "natural_minor"
ALT_TONIC_MIDI = 62             # D4
ALT_SCALE_INTERVALS = [0, 2, 3, 5, 7, 8, 10]

print(f"Primary: {KEY_NAME} {SCALE_NAME}, {BPM} BPM, {METER[0]}/{METER[1]}, {BARS} bars")
print(f"Variant: {ALT_KEY_NAME} {ALT_SCALE_NAME}, {ALT_BPM} BPM")

## Step 3 — Define the arrangement as MIDI events

Each melody event is `(beat_in_bar, scale_degree, duration_in_beats, velocity)`.
The bass follows a four-bar chord-root pattern and plays the root and fifth of each bar.

This is a deliberately small, hand-authored motif. It gives us a clean baseline for testing controls before asking a composition model to invent note sequences.

In [ ]:
# Eight melody notes per bar: eighth-note onsets on a 0.5-beat grid.
MELODY_PATTERN = [
    [(0.0, 0, 0.42, 92), (0.5, 2, 0.42, 76), (1.0, 4, 0.42, 84), (1.5, 2, 0.42, 72),
     (2.0, 3, 0.42, 86), (2.5, 4, 0.42, 78), (3.0, 2, 0.42, 74), (3.5, 1, 0.42, 70)],
    [(0.0, 4, 0.42, 88), (0.5, 5, 0.42, 80), (1.0, 6, 0.42, 84), (1.5, 5, 0.42, 74),
     (2.0, 4, 0.42, 86), (2.5, 2, 0.42, 76), (3.0, 3, 0.42, 78), (3.5, 2, 0.42, 70)],
    [(0.0, 0, 0.42, 90), (0.5, 2, 0.42, 76), (1.0, 4, 0.42, 84), (1.5, 6, 0.42, 78),
     (2.0, 5, 0.42, 86), (2.5, 3, 0.42, 74), (3.0, 4, 0.42, 80), (3.5, 2, 0.42, 72)],
    [(0.0, 1, 0.42, 84), (0.5, 3, 0.42, 76), (1.0, 5, 0.42, 82), (1.5, 4, 0.42, 74),
     (2.0, 2, 0.42, 84), (2.5, 1, 0.42, 72), (3.0, 0, 0.42, 80), (3.5, 0, 0.80, 92)],
]

# One chord root per bar: I, IV, V, iii in C major.
BASS_ROOT_DEGREES = [0, 3, 4, 2]

def degree_offset_semitones(degree, scale_intervals):
    """Map a signed diatonic scale degree to semitones relative to the tonic."""
    size = len(scale_intervals)
    if degree >= 0:
        octave, index = divmod(degree, size)
        return 12 * octave + scale_intervals[index]
    return -degree_offset_semitones(-degree, scale_intervals)

def build_events(tonic_midi, scale_intervals, bars=BARS):
    events = []
    for bar in range(bars):
        for beat_in_bar, degree, duration_beats, velocity in MELODY_PATTERN[bar]:
            pitch = tonic_midi + degree_offset_semitones(degree, scale_intervals)
            events.append({
                "part": "Lead", "bar": bar, "beat_in_bar": beat_in_bar,
                "start_beat": bar * METER[0] + beat_in_bar,
                "degree": degree, "pitch": pitch,
                "duration_beats": duration_beats, "velocity": velocity,
            })
        root_degree = BASS_ROOT_DEGREES[bar]
        for beat_in_bar, degree, duration_beats, velocity in [
            (0.0, root_degree, 1.75, 88),
            (2.0, root_degree + 4, 1.75, 76),
        ]:
            pitch = tonic_midi + degree_offset_semitones(degree, scale_intervals) - 12
            events.append({
                "part": "Bass", "bar": bar, "beat_in_bar": beat_in_bar,
                "start_beat": bar * METER[0] + beat_in_bar,
                "degree": degree, "pitch": pitch,
                "duration_beats": duration_beats, "velocity": velocity,
            })
    return events

events_c = build_events(KEY_TONIC_MIDI, SCALE_INTERVALS)
events_d = build_events(ALT_TONIC_MIDI, ALT_SCALE_INTERVALS)

print("C major event count:", len(events_c))
print("D natural minor event count:", len(events_d))
print("The two versions use the same beat positions and scale-degree pattern.")

## Step 4 — Export real MIDI files

The exported Standard MIDI Files contain separate Lead and Bass tracks, tempo, meter, instrument program numbers, note pitches, note-on times, note lengths, and velocities.

The program numbers are General MIDI identifiers for downstream players. The diagnostic WAV renderer in this notebook is not a GM SoundFont and does not claim those exact instrument timbres.

In [ ]:
def vlq(value):
    """Encode a non-negative MIDI variable-length quantity."""
    value = int(value)
    if value < 0:
        raise ValueError("MIDI delta times cannot be negative")
    buffer = [value & 0x7F]
    value >>= 7
    while value:
        buffer.insert(0, (value & 0x7F) | 0x80)
        value >>= 7
    return bytes(buffer)

def meta_event(kind, payload):
    return bytes([0xFF, kind]) + vlq(len(payload)) + payload

def write_track(events_for_track, track_name, channel, program=None, tempo_bpm=None, meter=None):
    timed = []
    if track_name:
        timed.append((0, -3, meta_event(0x03, track_name.encode("ascii"))))
    if tempo_bpm is not None:
        micros_per_beat = round(60_000_000 / tempo_bpm)
        timed.append((0, -2, meta_event(0x51, micros_per_beat.to_bytes(3, "big"))))
    if meter is not None:
        numerator, denominator = meter
        denominator_power = int(round(math.log2(denominator)))
        timed.append((0, -1, meta_event(0x58, bytes([numerator, denominator_power, 24, 8]))))
    if program is not None:
        timed.append((0, 0, bytes([0xC0 | channel, program])))

    for ev in events_for_track:
        start_tick = round(ev["start_beat"] * TICKS_PER_BEAT)
        end_tick = round((ev["start_beat"] + ev["duration_beats"]) * TICKS_PER_BEAT)
        pitch = int(ev["pitch"])
        velocity = int(np.clip(ev["velocity"], 1, 127))
        timed.append((start_tick, 1, bytes([0x90 | channel, pitch, velocity])))
        timed.append((end_tick, 0, bytes([0x80 | channel, pitch, 0])))

    timed.sort(key=lambda item: (item[0], item[1]))
    body = bytearray()
    previous_tick = 0
    for tick, _, data in timed:
        body.extend(vlq(tick - previous_tick))
        body.extend(data)
        previous_tick = tick
    body.extend(b"\x00\xFF\x2F\x00")
    return b"MTrk" + len(body).to_bytes(4, "big") + body

def write_midi(path, events, tempo_bpm, meter=METER):
    conductor = write_track([], "Conductor", 0, tempo_bpm=tempo_bpm, meter=meter)
    lead = write_track([e for e in events if e["part"] == "Lead"], "Lead", 0, program=4)
    bass = write_track([e for e in events if e["part"] == "Bass"], "Bass", 1, program=38)
    header = b"MThd" + (6).to_bytes(4, "big") + struct.pack(">HHH", 1, 3, TICKS_PER_BEAT)
    Path(path).write_bytes(header + conductor + lead + bass)
    return Path(path)

midi_c = write_midi(OUT_DIR / "C_major_96bpm.mid", events_c, BPM)
midi_d = write_midi(OUT_DIR / "D_natural_minor_120bpm.mid", events_d, ALT_BPM)
print("Saved:", midi_c)
print("Saved:", midi_d)

## Step 5 — Render separate audio layers

To keep this first experiment self-contained and license-transparent, the preview renderer below creates tones mathematically. It uses no downloaded samples or model weights.

This is intentionally a **control diagnostic**. A production version can swap in a cleared SoundFont or licensed virtual instrument while keeping the same MIDI arrangement and edit controls.

In [ ]:
# Each profile is an additive oscillator recipe plus a simple amplitude envelope.
TIMBRES = {
    "pluck": {"harmonics": [1.0, 0.42, 0.18, 0.08], "decay": 3.2, "attack": 0.006, "release": 0.08},
    "bell":  {"harmonics": [1.0, 0.58, 0.28, 0.16, 0.08], "decay": 1.15, "attack": 0.004, "release": 0.22},
    "warm":  {"harmonics": [1.0, 0.30, 0.10], "decay": 0.18, "attack": 0.035, "release": 0.16},
}

def midi_hz(pitch):
    return 440.0 * (2.0 ** ((pitch - 69) / 12.0))

def render_part(part_events, tempo_bpm, timbre_name, bars=BARS, sample_rate=SAMPLE_RATE):
    profile = TIMBRES[timbre_name]
    seconds_per_beat = 60.0 / tempo_bpm
    total_seconds = bars * METER[0] * seconds_per_beat + 0.35
    output = np.zeros(int(math.ceil(total_seconds * sample_rate)), dtype=np.float32)

    for ev in part_events:
        start_s = ev["start_beat"] * seconds_per_beat
        note_s = ev["duration_beats"] * seconds_per_beat
        release_s = profile["release"]
        n = int(math.ceil((note_s + release_s) * sample_rate))
        if n <= 0:
            continue
        t = np.arange(n, dtype=np.float32) / sample_rate
        attack = max(profile["attack"], 1.0 / sample_rate)
        attack_env = np.minimum(t / attack, 1.0)
        release_env = np.ones_like(t)
        release_start = int(note_s * sample_rate)
        if release_start < n:
            release_env[release_start:] = np.linspace(1.0, 0.0, n - release_start, endpoint=True)
        env = attack_env * release_env * np.exp(-profile["decay"] * t)

        frequency = midi_hz(ev["pitch"])
        wave = np.zeros_like(t)
        for harmonic_index, weight in enumerate(profile["harmonics"], start=1):
            wave += weight * np.sin(2 * np.pi * frequency * harmonic_index * t)
        wave *= env * (ev["velocity"] / 127.0) * 0.18

        begin = int(round(start_s * sample_rate))
        finish = min(begin + n, len(output))
        if finish > begin:
            output[begin:finish] += wave[:finish - begin]

    return output

def mix_and_normalize(track_waves):
    length = max(map(len, track_waves.values()))
    mix = np.zeros(length, dtype=np.float32)
    for wave in track_waves.values():
        mix[:len(wave)] += wave
    peak = float(np.max(np.abs(mix))) if len(mix) else 0.0
    if peak > 0:
        mix = mix * (0.95 / peak)
    return mix

def save_float_wav(path, wave, sample_rate=SAMPLE_RATE):
    pcm = np.clip(wave, -1.0, 1.0)
    write_wav(str(path), sample_rate, (pcm * 32767).astype(np.int16))
    return Path(path)

## Step 6 — Create controlled versions

The first pair has **identical MIDI notes and rhythm**; only the Lead timbre changes. The second version uses the same scale-degree motif in D natural minor at 120 BPM. This separates:
- timbre change from note change;
- tempo change from note-sequence change;
- key/scale change from arrangement shape.

In [ ]:
def render_arrangement(events, tempo_bpm, lead_timbre="pluck"):
    lead_events = [e for e in events if e["part"] == "Lead"]
    bass_events = [e for e in events if e["part"] == "Bass"]
    lead = render_part(lead_events, tempo_bpm, lead_timbre)
    bass = render_part(bass_events, tempo_bpm, "warm")
    mix = mix_and_normalize({"Lead": lead, "Bass": bass})
    return {"Lead": lead, "Bass": bass, "Mix": mix}

render_c_pluck = render_arrangement(events_c, BPM, "pluck")
render_c_bell = render_arrangement(events_c, BPM, "bell")
render_d_pluck = render_arrangement(events_d, ALT_BPM, "pluck")

outputs = {}
for label, rendered in [
    ("C_major_96_pluck", render_c_pluck),
    ("C_major_96_bell_lead", render_c_bell),
    ("D_minor_120_pluck", render_d_pluck),
]:
    for part_name, wave in rendered.items():
        out_path = save_float_wav(OUT_DIR / f"{label}_{part_name.lower()}.wav", wave)
        outputs[f"{label}_{part_name}"] = out_path

print("Created WAV files:")
for name, path in outputs.items():
    print(f"- {name}: {path.name}")

print("\nC major, pluck lead:")
display(Audio(render_c_pluck["Mix"], rate=SAMPLE_RATE))
print("Same MIDI, bell lead:")
display(Audio(render_c_bell["Mix"], rate=SAMPLE_RATE))
print("Same motif shape, D natural minor at 120 BPM:")
display(Audio(render_d_pluck["Mix"], rate=SAMPLE_RATE))

## Step 7 — Inspect the exact note and timing data

The table and piano roll expose the actual notes, beats, and lengths. These are not estimated from the rendered audio; they are the source events used to write the MIDI and render the WAV.

In [ ]:
def midi_note_name(pitch):
    names = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
    octave = (pitch // 12) - 1
    return f"{names[pitch % 12]}{octave}"

def show_event_summary(events, limit=20):
    for ev in events[:limit]:
        print(
            f'{ev["part"]:4s} | bar {ev["bar"] + 1} | beat {ev["beat_in_bar"]:>3} '
            f'| {midi_note_name(ev["pitch"]):4s} | duration {ev["duration_beats"]:.2f} beats '
            f'| velocity {ev["velocity"]}'
        )

show_event_summary(events_c, limit=24)

lead = [e for e in events_c if e["part"] == "Lead"]
plt.figure(figsize=(12, 4))
for ev in lead:
    plt.plot(
        [ev["start_beat"], ev["start_beat"] + ev["duration_beats"]],
        [ev["pitch"], ev["pitch"]],
        linewidth=5,
        solid_capstyle="butt",
    )
plt.xticks(np.arange(0, BARS * METER[0] + 0.01, 1))
plt.xlabel("Beat")
plt.ylabel("MIDI pitch")
plt.title(f"Lead piano roll — {KEY_NAME} {SCALE_NAME}, {BPM} BPM")
plt.grid(True, alpha=0.25)
plt.show()

## Step 8 — Verify the declared controls

These checks summarize whether the MIDI event list obeys the settings we entered. We check that every note belongs to the chosen scale and every onset lands on the intended eighth-note grid.

This is a control audit of the **explicit MIDI representation**, not a subjective music-quality score.

In [ ]:
def control_report(events, tonic_midi, scale_intervals, bpm, name):
    allowed_pitch_classes = {(tonic_midi + interval) % 12 for interval in scale_intervals}
    out_of_scale = [e for e in events if e["pitch"] % 12 not in allowed_pitch_classes]
    off_grid = [e for e in events if abs(e["start_beat"] * 2 - round(e["start_beat"] * 2)) > 1e-8]
    duration_s = BARS * METER[0] * 60.0 / bpm
    report = {
        "version": name,
        "tempo_bpm": bpm,
        "meter": f"{METER[0]}/{METER[1]}",
        "event_count": len(events),
        "out_of_scale_notes": len(out_of_scale),
        "off_eighth_grid_onsets": len(off_grid),
        "musical_duration_s": round(duration_s, 3),
        "control_pass": not out_of_scale and not off_grid,
    }
    return report

reports = [
    control_report(events_c, KEY_TONIC_MIDI, SCALE_INTERVALS, BPM, "C major"),
    control_report(events_d, ALT_TONIC_MIDI, ALT_SCALE_INTERVALS, ALT_BPM, "D natural minor"),
]
print(json.dumps(reports, indent=2))
assert all(r["control_pass"] for r in reports), "At least one event violates the chosen scale or rhythm grid."

## Step 9 — Download the editable project bundle

The ZIP includes:
- two editable MIDI files;
- separate Lead and Bass WAV previews;
- full-mix WAV previews;
- a JSON manifest containing the musical controls and exact note events.

The WAV preview timbres are synthesized by the notebook's small mathematical renderer. For a polished sound, the next step is to choose and audit a commercially usable instrument source, then render the same MIDI through it.

In [ ]:
manifest = {
    "experiment": "midi_exact_control_pilot_v1",
    "renderer": "original additive oscillator diagnostic; no sampled audio",
    "primary": {
        "key": f"{KEY_NAME} {SCALE_NAME}",
        "tempo_bpm": BPM,
        "meter": list(METER),
        "midi_file": midi_c.name,
        "events": events_c,
    },
    "variant": {
        "key": f"{ALT_KEY_NAME} {ALT_SCALE_NAME}",
        "tempo_bpm": ALT_BPM,
        "meter": list(METER),
        "midi_file": midi_d.name,
        "events": events_d,
    },
    "control_reports": reports,
    "outputs": {name: path.name for name, path in outputs.items()},
}
manifest_path = OUT_DIR / "control_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

bundle_path = Path("/content/midi_exact_control_pilot.zip")
with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT_DIR.iterdir()):
        if path.is_file():
            archive.write(path, arcname=path.name)

print("Bundle ready:", bundle_path, "| bytes:", bundle_path.stat().st_size)
from google.colab import files
files.download(str(bundle_path))

## How to interpret this pilot

If the report shows zero out-of-scale notes and zero off-grid onsets, we have verified the core **controllability** path: notes and rhythm come from explicit data, and instrument tracks remain separable.

We have not yet proven that the diagnostic renderer sounds commercially polished, that an AI can compose good arrangements from only a few settings, or that Stable Audio follows exact MIDI events. Those are separate next steps:
1. render this same MIDI with a license-cleared higher-quality instrument;
2. let a composition model propose MIDI, then quantize/validate it against the selected scale, meter, and chord plan;
3. compare that controlled MIDI render with the Stable Audio track and inpainted edit.